<a href="https://colab.research.google.com/github/equipe31ccpj/Solucoes-Renovaveis-CitizenAssistAI/blob/main/SERS_GS2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Citizen Assist AI — Análise de Consumo e Alternativas Energéticas (SERS)
**Público-Alvo:** Terceira Idade (residências de 1 a 2 moradores).  
**Objetivo:** Analisar o perfil de consumo residencial, comparar alternativas de economia e geração renovável (Eficiência Energética vs. Energia Solar) e integrar os resultados às respostas do assistente digital.

### Origem dos Dados e Premissas
* **Dataset:** *Household Energy Consumption* (Kaggle — 90.000 registros diários de 2025).
* **Recorte:** Filtradas 30.437 ocorrências de casas com 1 a 2 habitantes.
* **Premissas de Simulação:** Período mensal de 30 dias; tarifa residencial média de **R$ 0,85/kWh** (ANEEL); fator de emissão de **0,085 kg CO₂/kWh** (MCTI).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Carrega o dataset CSV
arquivo = 'household_energy_consumption_2025.csv'
df = pd.read_csv("/content/household_energy_consumption.csv")

print("=== INFORMAÇÕES INICIAIS DO DATASET ===")
print(f"Total de registros: {len(df):,}")
print(f"Valores ausentes por coluna:\n{df.isnull().sum()}\n")




FileNotFoundError: [Errno 2] No such file or directory: '/content/household_energy_consumption.csv'

In [ ]:
# 2. Filtra residências com perfil típico da terceira idade
df_idosos = df[df['Household_Size'] <= 2].copy()

# Calcula a porcentagem de consumo que ocorre no horário de pico
df_idosos['Pct_Horario_Pico'] = (df_idosos['Peak_Hours_Usage_kWh'] / df_idosos['Energy_Consumption_kWh']) * 100

print(f"Registros filtrados (residências de 1 a 2 pessoas): {len(df_idosos):,}")
df_idosos.head()

## 1. Análise Exploratória de Consumo e Horários de Pico
Nesta etapa, calculamos o consumo médio diário e mensal em **kWh**, identificamos o peso do gasto no horário de pico e avaliamos o impacto da temperatura ambiente e do uso de ar-condicionado.


In [ ]:
# 1. Estatísticas descritivas do consumo diário (kWh)
print("=== ESTATÍSTICAS DE CONSUMO DIÁRIO (kWh) - PERFIL 1 A 2 MORADORES ===")
print(df_idosos[['Energy_Consumption_kWh', 'Peak_Hours_Usage_kWh', 'Avg_Temperature_C']].describe().round(2))

# Premissa de referência mensal (30 dias)
consumo_diario_medio = df_idosos['Energy_Consumption_kWh'].mean()
consumo_mensal_medio = consumo_diario_medio * 30
pico_diario_medio = df_idosos['Peak_Hours_Usage_kWh'].mean()

print(f"\nConsumo Diário Médio: {consumo_diario_medio:.2f} kWh/dia")
print(f"Consumo Mensal Estimado (30 dias): {consumo_mensal_medio:.2f} kWh/mês")
print(f"Média gasta apenas no Horário de Pico: {pico_diario_medio:.2f} kWh/dia ({(pico_diario_medio/consumo_diario_medio)*100:.1f}% do total)")


In [ ]:
# 2. Gráficos de Análise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Impacto do Ar-Condicionado no Consumo
sns.barplot(data=df_idosos, x='Has_AC', y='Energy_Consumption_kWh', ax=axes[0], palette=['#5cb85c', '#d9534f'])
axes[0].set_title('Consumo Médio Diário: Com vs. Sem Ar-Condicionado', fontweight='bold')
axes[0].set_xlabel('Possui Ar-Condicionado (Has_AC)')
axes[0].set_ylabel('Consumo Médio (kWh/dia)')

plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 2: Relação entre Temperatura e Consumo de Energia
sns.scatterplot(data=df_idosos.sample(1000, random_state=42), x='Avg_Temperature_C', y='Energy_Consumption_kWh', hue='Has_AC', alpha=0.6, ax=axes[1])
axes[1].set_title('Influência da Temperatura no Consumo Energético', fontweight='bold')
axes[1].set_xlabel('Temperatura Média (°C)')
axes[1].set_ylabel('Consumo Diário (kWh)')

plt.tight_layout()
plt.show()


## 2. Comparação de Alternativas: Eficiência Energética vs. Energia Solar
Comparação financeira, de viabilidade (retorno do investimento) e de impacto ambiental (redução de CO₂) para o mesmo consumo mensal médio de **142,65 kWh/mês**:
* **Alternativa 1 (Eficiência Energética):** Redução estimada de 18% com monitoramento visual (protótipo Arduino) e ajuste de hábitos nos horários de pico.
* **Alternativa 2 (Fonte Renovável - Energia Solar):** Suprimento estimado de 85% do consumo via painéis próprios ou energia solar compartilhada.


In [ ]:
# === PREMISSAS DA SIMULAÇÃO (Documentar no README e no Colab) ===
tarifa_kwh = 0.85           # Premissa: Tarifa média residencial no Brasil (R$ 0,85 / kWh - Fonte: ANEEL)
fator_co2_kg_kwh = 0.085    # Premissa: Fator de emissão do Sistema Interligado Nacional (kg CO2/kWh - Fonte: MCTI)

# Cenário Base (Sem intervenção)
custo_mensal_atual = consumo_mensal_medio * tarifa_kwh
co2_mensal_atual = consumo_mensal_medio * fator_co2_kg_kwh


In [ ]:
# --- ALTERNATIVA 1: EFICIÊNCIA ENERGÉTICA (Monitoramento Tinkercad + Mudança de Hábitos) ---
# Redução estimada de 18% ao evitar uso de climatização/aparelhos pesados nos horários de pico
reducao_eficiencia_pct = 0.18
consumo_alt1 = consumo_mensal_medio * (1 - reducao_eficiencia_pct)
custo_alt1 = consumo_alt1 * tarifa_kwh
economia_mensal_alt1 = custo_mensal_atual - custo_alt1
investimento_alt1 = 150.00  # Custo estimado do kit de monitoramento / lâmpadas LED
payback_alt1_meses = investimento_alt1 / economia_mensal_alt1
co2_evitado_alt1 = (consumo_mensal_medio - consumo_alt1) * fator_co2_kg_kwh

In [ ]:
# --- ALTERNATIVA 2: FONTE RENOVÁVEL (Energia Solar Fotovoltaica / Geração Distribuída) ---
# Suprimento de 85% do consumo mensal (mantendo apenas taxa mínima da concessionária)
reducao_solar_pct = 0.85
consumo_rede_alt2 = consumo_mensal_medio * (1 - reducao_solar_pct)
custo_alt2 = consumo_rede_alt2 * tarifa_kwh
economia_mensal_alt2 = custo_mensal_atual - custo_alt2
investimento_alt2 = 9500.00 # Investimento médio estimado para sistema solar residencial pequeno
payback_alt2_meses = investimento_alt2 / economia_mensal_alt2
co2_evitado_alt2 = (consumo_mensal_medio - consumo_rede_alt2) * fator_co2_kg_kwh

In [ ]:
# Tabela Comparativa Final
comparacao = pd.DataFrame({
    'Cenário': ['Atual (Sem Mudanças)', 'Alt 1: Eficiência Energética', 'Alt 2: Energia Solar (Renovável)'],
    'Consumo da Rede (kWh/mês)': [consumo_mensal_medio, consumo_alt1, consumo_rede_alt2],
    'Conta de Luz (R$/mês)': [custo_mensal_atual, custo_alt1, custo_alt2],
    'Economia Mensal (R$)': [0.0, economia_mensal_alt1, economia_mensal_alt2],
    'Investimento Inicial (R$)': [0.0, investimento_alt1, investimento_alt2],
    'Retorno / Payback (Meses)': [0.0, payback_alt1_meses, payback_alt2_meses],
    'CO2 Evitado (kg/mês)': [0.0, co2_evitado_alt1, co2_evitado_alt2]
})

print("=== TABELA COMPARATIVA DE ALTERNATIVAS (BASE MENSAL) ===")
display(comparacao.round(2))


## 3. Integração ao Citizen Assist AI e Teste Interativo
Abaixo, os cálculos validados na análise alimentam diretamente o módulo de orientação do assistente, traduzindo os dados em recomendações acessíveis, com indicação de fontes e limitações das estimativas.


In [ ]:
def citizen_assist_energia(consumo_usuario_kwh=None, tem_ar_condicionado=False, duvida="geral"):
    """
    Módulo de orientação energética do Citizen Assist AI para a Terceira Idade.
    Usa os resultados reais da análise do dataset Household Energy Consumption.
    """
    # Se o idoso não souber o consumo exato, usa a média encontrada no dataset para 1-2 moradores
    if consumo_usuario_kwh is None:
        consumo_ref = consumo_mensal_medio
        origem_dado = f"média estimada de {consumo_ref:.0f} kWh/mês para residências de 1 a 2 pessoas"
    else:
        consumo_ref = consumo_usuario_kwh
        origem_dado = f"seu consumo informado de {consumo_ref:.0f} kWh/mês"

    conta_estimada = consumo_ref * tarifa_kwh

    # Classificação alinhada ao semáforo do protótipo Tinkercad (Verde / Amarelo / Vermelho)
    if consumo_ref <= 180:
        status_semaforo = "VERDE (Consumo Baixo e Equilibrado)"
    elif consumo_ref <= 350:
        status_semaforo = "AMARELO (Atenção: Consumo Moderado)"
    else:
        status_semaforo = "VERMELHO (Alerta: Consumo Elevado — Hora de Economizar!)"

    # Cálculos personalizados baseados nas taxas validadas no Colab
    econ_eficiencia = conta_estimada * reducao_eficiencia_pct
    econ_solar = conta_estimada * reducao_solar_pct
    co2_evitado_solar = (consumo_ref * reducao_solar_pct) * fator_co2_kg_kwh

    print("=" * 65)
    print("CITIZEN ASSIST AI — ORIENTAÇÃO DE ENERGIA E SUSTENTABILIDADE")
    print("=" * 65)
    print(f"Olá! Analisei {origem_dado} (gasto estimado de R$ {conta_estimada:.2f}/mês).")
    print(f"Status do seu Monitor Inteligente: {status_semaforo}\n")

    if duvida == "economia" or duvida == "geral":
        print("ALTERNATIVA 1: EFICIÊNCIA ENERGÉTICA (Sem grandes gastos)")
        print(f"• Como funciona: Evitar aparelhos pesados no horário de pico (18h às 21h) e ajustar o uso diário.")
        if tem_ar_condicionado:
            print("• Dica especial (Ar-Condicionado): Nossa análise mostrou que o ar-condicionado eleva muito o gasto. Mantenha em 23°C e limpe os filtros.")
        print(f"• Economia estimada: Cerca de R$ {econ_eficiencia:.2f} por mês (redução de ~18%).")
        print("• Vantagem: Custo quase zero e retorno imediato para o seu orçamento.\n")

    if duvida == "solar" or duvida == "geral":
        print("ALTERNATIVA 2: ENERGIA SOLAR (Fonte Renovável)")
        print("• Como funciona: Instalação de painéis solares ou adesão a planos de energia solar compartilhada (sem obras no telhado).")
        print(f"• Economia estimada: Até R$ {econ_solar:.2f} por mês na conta de luz (redução de até 85%).")
        print(f"• Benefício ambiental: Evita a emissão de {co2_evitado_solar:.1f} kg de CO2 por mês.\n")

    print("CONDIÇÕES E LIMITAÇÕES (Transparência):")
    print("• Os valores são estimativas baseadas na tarifa média de R$ 0,85/kWh (ANEEL) e no dataset Household Energy Consumption (Kaggle).")
    print("• O valor real pode variar conforme a bandeira tarifária da sua região, clima e aparelhos da casa.")
    print("=" * 65)

# --- TESTANDO O ASSISTENTE NO COLAB (Simulação de Interação) ---
# Exemplo 1: Idoso que mora sozinho, tem ar-condicionado e gasta 420 kWh/mês
citizen_assist_energia(consumo_usuario_kwh=420, tem_ar_condicionado=True, duvida="geral")


In [ ]:
print(">>> TESTE 1: Dúvida sobre Economia sem saber o consumo exato <<<")
citizen_assist_energia(consumo_usuario_kwh=None, tem_ar_condicionado=False, duvida="economia")

print("\n>>> TESTE 2: Dúvida sobre Energia Solar com consumo elevado <<<")
citizen_assist_energia(consumo_usuario_kwh=420, tem_ar_condicionado=True, duvida="solar")

print("\n>>> TESTE 3: Visão Geral (Consumo Moderado) <<<")
citizen_assist_energia(consumo_usuario_kwh=250, tem_ar_condicionado=True, duvida="geral")
